# Exploratory Data Analysis (EDA)

**Objectif :** Analyser et comprendre le dataset COVID-19 avant le modeling.

Ce notebook utilise la classe **EDAAnalyzer** pour une analyse structurée et réutilisable.

## 1. Setup et Imports

In [ ]:
# Setup
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
pd.set_option('display.max_columns', 100)
sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
# Import de la classe EDAAnalyzer
from src.eda import EDAAnalyzer
from src.utils import load_data
from src.config import TARGET_FEATURE

print(f"Target feature: {TARGET_FEATURE}")

## 2. Chargement des Données et Initialisation

In [ ]:
# Charger les données
df = load_data()
print(f"Dataset chargé avec succès!")
print(f"Shape: {df.shape}")

In [ ]:
# Initialiser l'analyseur EDA
eda = EDAAnalyzer(df)
print("EDAAnalyzer initialisé!")

In [ ]:
# Aperçu des premières lignes
df.head()

## 3. Analyse de la Structure

In [ ]:
# Informations de base sur le dataset
shape_info = eda.analyze_shape()

## 4. Analyse des Valeurs Manquantes

In [ ]:
# Analyser les valeurs manquantes
missing_rate = eda.analyze_missing_values(top_n=20)

In [ ]:
# Visualiser avec heatmap
eda.plot_missing_heatmap(figsize=(20, 10))

## 5. Analyse de la Target

In [ ]:
# Distribution de la target
target_info = eda.analyze_target()

In [ ]:
# Visualisation de la distribution
eda.plot_target_distribution(figsize=(14, 5))

## 6. Identification des Groupes de Features

In [ ]:
# Identifier les groupes blood et viral
blood_columns, viral_columns = eda.identify_feature_groups()

print(f"\nBlood features ({len(blood_columns)}):")
print(blood_columns[:5], "...")

print(f"\nViral features ({len(viral_columns)}):")
print(viral_columns)

## 7. Distribution des Features

In [ ]:
# Distribution des blood features (8 premiers)
eda.plot_feature_distributions(blood_columns[:8], n_cols=4, figsize=(16, 8))

In [ ]:
# Distribution des viral features
eda.plot_feature_distributions(viral_columns, n_cols=4, figsize=(16, 8))

## 8. Comparaison par Target

In [ ]:
# Comparer les distributions entre positifs et négatifs
eda.compare_distributions_by_target(blood_columns[:6], figsize=(14, 10))

## 9. Corrélations

In [ ]:
# Matrice de corrélation pour blood features
eda.plot_correlation_matrix(blood_columns[:12], figsize=(14, 12))

## 10. Tests Statistiques

In [ ]:
# Tests t pour identifier les features significatives
significant_features = eda.statistical_tests(blood_columns, alpha=0.02)

## 11. Résumé et Conclusions

In [ ]:
# Générer le rapport complet
summary = eda.generate_summary_report()

print("\n" + "="*70)
print("RÉSUMÉ DE L'EDA")
print("="*70)

print(f"\nDataset:")
print(f"  - Lignes: {summary['shape_info']['n_rows']:,}")
print(f"  - Colonnes: {summary['shape_info']['n_columns']}")
print(f"  - Mémoire: {summary['shape_info']['memory_mb']:.2f} MB")

print(f"\nValeurs manquantes:")
print(f"  - Colonnes avec >90% NaN: {summary['missing_summary']['n_cols_90pct']}")
print(f"  - Colonnes avec >50% NaN: {summary['missing_summary']['n_cols_50pct']}")
print(f"  - Colonnes sans NaN: {summary['missing_summary']['n_cols_no_missing']}")

print(f"\nTarget distribution:")
for cat, pct in summary['target_distribution']['percentages'].items():
    print(f"  - {cat}: {pct:.1f}%")

print(f"\nGroupes de features:")
print(f"  - Blood features: {summary['feature_groups']['blood_features']}")
print(f"  - Viral features: {summary['feature_groups']['viral_features']}")
print(f"  - Features significatives (p<0.02): {len(significant_features)}")

### Conclusions

#### Points clés:

1. **Dataset déséquilibré:**
   - ~90% de cas négatifs, ~10% de cas positifs
   - Nécessite une attention particulière lors du modeling (stratification, métriques adaptées)

2. **Valeurs manquantes:**
   - Beaucoup de features avec >90% de NaN (tests non effectués)
   - Deux groupes principaux: blood features (88-90% NaN) et viral features (75-88% NaN)
   - Stratégie de preprocessing critique pour le succès du modèle

3. **Groupes de features:**
   - Blood features: résultats d'analyses sanguines
   - Viral features: tests pour différents virus respiratoires
   - Possibilité de feature engineering avec les tests viraux

4. **Features significatives:**
   - Plusieurs blood features montrent des différences statistiquement significatives entre positifs et négatifs
   - Ces features seront importantes pour la prédiction

#### Prochaines étapes:
- Preprocessing: sélection de features, encodage, imputation
- Feature engineering: créer 'est malade' à partir des tests viraux
- Modeling avec focus sur le recall (détecter les cas positifs)